# NoSQL en Databricks · 03 · Desafío y entrega
Duración: 35–45 minutos. Resolvé las consignas y explicá tus decisiones de modelado.
Ejecutá previamente 00, 01 y 02. Las celdas TODO son deliberadamente incompletas.
El control final se ejecuta al seleccionar `validar_entrega = si`.

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "test", ["test", "small"], "02 - Escala NoSQL")
dbutils.widgets.dropdown("validar_entrega", "no", ["no", "si"], "03 - Validar resolución")

In [ ]:
%run ../common/config

In [ ]:
config = CourseConfig(
    spark.sql("SELECT current_catalog()").first()[0],
    dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"),
)
repetitions = {"test": 1, "small": 100}[config.scale]
required = ["nosql_orders", "nosql_raw_orders", "nosql_quarantine",
            "nosql_customers", "nosql_products"]
missing = [name for name in required
           if not spark.catalog.tableExists(qualified_table(config, name))]
assert not missing, f"Faltan {missing}. Ejecutá 00_setup_datos con el mismo student_id."
assert spark.table(qualified_table(config, "nosql_orders")).count() == 12 * repetitions,             "La escala no coincide con 00_setup_datos."
spark.sql(f"USE CATALOG {quote_identifier(config.catalog)}")
spark.sql(f"USE SCHEMA {quote_identifier(config.schema)}")
print(f"Laboratorio: {config.catalog}.{config.schema}; escala {config.scale}")

In [ ]:
assert spark.catalog.tableExists("nosql_order_items"), "Ejecutá 02_consultas_analiticas primero."

## A. Ventas por categoría
Creá una vista temporal `nosql_entrega_ventas` con una fila por categoría y estas columnas:
`category` STRING, `units` BIGINT y `amount` DECIMAL(18,2).
Usá solamente pedidos pagados y los precios históricos de sus líneas. No uses el catálogo actual.
Explicá por qué el total del pedido no es el importe correcto para sumar por categoría.
Las vistas de A y B deben tener exactamente las columnas y tipos indicados; se calculan sobre la base original, antes de la evolución de C.


In [ ]:
# TODO: spark.sql("CREATE OR REPLACE TEMP VIEW nosql_entrega_ventas AS ...")

## B. Alertas
Creá una vista temporal `nosql_entrega_alertas` con `order_id` STRING, `total` DECIMAL(12,2)
y `risk_score` DOUBLE. Aplicá la regla: paid, total >= 1000, risk.score >= 0.8.
Cada pedido debe aparecer una sola vez. Justificá por qué la salida indica sospecha y no fraude confirmado.

In [ ]:
# TODO: spark.sql("CREATE OR REPLACE TEMP VIEW nosql_entrega_alertas AS ...")

## C. Evolución del documento
Trabajá sobre la copia `nosql_orders_challenge`. La celda siguiente la reconstruye desde la base.
Construí `new_order` como diccionario Python con estos requisitos:
- `order_id = 'O_NUEVO_001'`, `schema_version = 3`, `customer_id = 'C0001'`.
- `status = 'paid'`, `currency = 'USD'`, `total = 60` y `risk.score = 0.05`.
- Una línea: producto P005, categoría libros, cantidad 2 y precio histórico 30.
- `customer_snapshot`, `shipping`, `payment` y `created_at`, siguiendo el modelo original.
- Un objeto nuevo `gift`, con un mensaje y un booleano `wrapped`.

Incorporá el documento usando el helper provisto. Su MERGE evita duplicados si repetís la incorporación.
Consultá `gift` junto con los pedidos previos: ¿qué devuelve para las versiones anteriores?
La tabla física debe conservar las mismas columnas, y los documentos previos, el mismo contenido.
Los campos textuales requeridos deben estar presentes y no vacíos; created_at debe convertirse a TIMESTAMP y gift.wrapped debe ser un booleano JSON, no el texto "true".


In [ ]:
spark.sql("""
CREATE OR REPLACE TABLE nosql_orders_challenge
USING DELTA AS SELECT * FROM nosql_orders
""")
print("Copia de trabajo lista. Esta celda reinicia solamente nosql_orders_challenge.")

In [ ]:
# TODO: reemplazá None por tu documento. No modifiques los documentos originales.
new_order = None

In [ ]:
import json
if new_order is None:
    print("Pendiente: completá new_order en la celda anterior.")
else:
    assert new_order.get("order_id") == "O_NUEVO_001"
    assert new_order.get("schema_version") == 3
    assert isinstance(new_order.get("gift"), dict)
    new_json = json.dumps(new_order, ensure_ascii=False, allow_nan=False)
    spark.createDataFrame([(new_json,)], "raw_json STRING").createOrReplaceTempView("nosql_new_order_raw")
    spark.sql("""
    MERGE INTO nosql_orders_challenge AS target
    USING (
      SELECT try_variant_get(parse_json(raw_json), '$.order_id', 'string') AS order_id,
             raw_json, parse_json(raw_json) AS doc,
             'desafio_manual' AS source_file, current_timestamp() AS ingested_at
      FROM nosql_new_order_raw
    ) AS source
    ON target.order_id = source.order_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """)
    display(spark.sql("SELECT order_id, doc:gift AS gift FROM nosql_orders_challenge WHERE order_id = 'O_NUEVO_001'"))

In [ ]:
%sql
SELECT order_id, doc:schema_version::INT AS version, doc:gift AS gift
FROM nosql_orders_challenge
WHERE order_id IN ('O000001', 'O000009', 'O_NUEVO_001') ORDER BY order_id

## D. Explicación conceptual
Escribí en la entrega:
1. Una decisión de embeber y otra de referenciar, con su motivo y costo.
2. Las diferencias observadas entre cupón ausente, null explícito y texto vacío.
3. Una regla adicional de calidad y qué problema detectaría.
4. Qué sucede al agregar `gift` sin ALTER TABLE y qué debe cambiar en la aplicación consumidora.
5. Qué capacidades necesitarías verificar en una base documental real: índices, unicidad,
   concurrencia, particionamiento y consistencia. ¿Cuáles no evaluamos aquí?

## Control final de resolución
Seleccioná `validar_entrega = si` y ejecutá esta celda después de resolver A, B y C.
Las vistas temporales viven en la sesión de este notebook: crealas acá.

In [ ]:
if dbutils.widgets.get("validar_entrega") != "si":
    print("Validación pendiente: resolvé los TODO y seleccioná validar_entrega = si.")
else:
    from decimal import Decimal

    for name in ["nosql_entrega_ventas", "nosql_entrega_alertas", "nosql_orders_challenge"]:
        assert spark.catalog.tableExists(name), f"Falta crear {name}"

    def validate_delivery_schema(frame, expected_types, name):
        assert dict(frame.dtypes) == expected_types, f"Revisá las columnas y tipos de {name}."

    def validate_evolved_order(new, new_lines):
        from decimal import Decimal

        assert new is not None, "Falta O_NUEVO_001."
        assert new.version == 3 and new.gift_type and new.gift_type.startswith("OBJECT<")
        assert new.total == Decimal("60.00") and new.status == "paid" and new.currency == "USD"
        assert new.customer_id == "C0001" and new.items_type and new.items_type.startswith("ARRAY<")
        assert new.risk_score == 0.05
        assert new.gift_message_type == "STRING" and new.gift_message, "gift.message debe ser texto no vacío."
        assert new.gift_wrapped_type == "BOOLEAN" and new.gift_wrapped is not None, "gift.wrapped debe ser un booleano JSON."
        assert new.created_at is not None, "created_at debe ser una fecha válida."
        assert new.snapshot_name_type == "STRING" and new.snapshot_name, "Falta el nombre del snapshot."
        assert new.snapshot_country_type == "STRING" and new.snapshot_country, "Falta el país del snapshot."
        assert new.shipping_country_type == "STRING" and new.shipping_country, "Falta el país de envío."
        assert new.shipping_city_type == "STRING" and new.shipping_city, "Falta la ciudad de envío."
        assert new.payment_method_type == "STRING" and new.payment_method, "Falta el medio de pago."
        assert len(new_lines) == 1, "El pedido nuevo debe tener una línea."
        line = new_lines[0]
        assert (line.product_id, line.category, line.quantity, line.price) == ("P005", "libros", 2, Decimal("30.00"))

    validate_delivery_schema(spark.table("nosql_entrega_ventas"),
        {"category": "string", "units": "bigint", "amount": "decimal(18,2)"}, "nosql_entrega_ventas")
    validate_delivery_schema(spark.table("nosql_entrega_alertas"),
        {"order_id": "string", "total": "decimal(12,2)", "risk_score": "double"}, "nosql_entrega_alertas")

    sales_rows = spark.sql("SELECT category, units, amount FROM nosql_entrega_ventas").collect()
    assert len(sales_rows) == 3, "Debe haber una fila por categoría."
    actual_sales = {row.category: (row.units, row.amount) for row in sales_rows}
    expected_sales = {
        "tecnologia": (10 * repetitions, Decimal("4600.00") * repetitions),
        "hogar": (13 * repetitions, Decimal("530.00") * repetitions),
        "libros": (4 * repetitions, Decimal("120.00") * repetitions),
    }
    assert actual_sales == expected_sales, f"Ventas incorrectas: {actual_sales}"
    alert_rows = spark.sql("SELECT order_id, total, risk_score FROM nosql_entrega_alertas").collect()
    expected_alerts = {
        f"O{block * 12 + local_id:06d}": (Decimal(total), score)
        for block in range(repetitions)
        for local_id, total, score in [(5, "2000.00", 0.95), (9, "1030.00", 0.92)]
    }
    assert len(alert_rows) == len(expected_alerts), "Cantidad de alertas incorrecta o duplicados."
    assert {r.order_id: (r.total, r.risk_score) for r in alert_rows} == expected_alerts, "Revisá estado, total, puntaje y tipos de las alertas."

    assert spark.table("nosql_orders_challenge").count() == 12 * repetitions + 1
    assert spark.table("nosql_orders_challenge").dtypes == spark.table("nosql_orders").dtypes, "La evolución debe conservar las columnas y tipos físicos de la tabla."
    assert spark.sql("SELECT COUNT(DISTINCT order_id) FROM nosql_orders_challenge").first()[0] == 12 * repetitions + 1
    changed = spark.sql("""
    SELECT COUNT(*) FROM nosql_orders AS original
    LEFT JOIN nosql_orders_challenge AS revised ON original.order_id = revised.order_id
    WHERE revised.order_id IS NULL OR NOT (original.raw_json <=> revised.raw_json)
       OR NOT (to_json(original.doc) <=> to_json(revised.doc))
    """).first()[0]
    assert changed == 0, "La evolución debe conservar todos los documentos anteriores."
    new = spark.sql("""
    SELECT try_variant_get(doc, '$.schema_version', 'int') AS version,
           try_variant_get(doc, '$.total', 'decimal(12,2)') AS total,
           try_variant_get(doc, '$.status', 'string') AS status,
           try_variant_get(doc, '$.currency', 'string') AS currency,
           try_variant_get(doc, '$.customer_id', 'string') AS customer_id,
           try_variant_get(doc, '$.risk.score', 'double') AS risk_score,
           schema_of_variant(doc:gift) AS gift_type,
           try_variant_get(doc, '$.gift.message', 'string') AS gift_message,
           try_variant_get(doc, '$.gift.wrapped', 'boolean') AS gift_wrapped,
           schema_of_variant(doc:items) AS items_type,
           schema_of_variant(doc:gift.message) AS gift_message_type,
           schema_of_variant(doc:gift.wrapped) AS gift_wrapped_type,
           try_variant_get(doc, '$.created_at', 'timestamp') AS created_at,
           try_variant_get(doc, '$.customer_snapshot.name', 'string') AS snapshot_name,
           schema_of_variant(doc:customer_snapshot.name) AS snapshot_name_type,
           try_variant_get(doc, '$.customer_snapshot.country', 'string') AS snapshot_country,
           schema_of_variant(doc:customer_snapshot.country) AS snapshot_country_type,
           try_variant_get(doc, '$.shipping.country', 'string') AS shipping_country,
           schema_of_variant(doc:shipping.country) AS shipping_country_type,
           try_variant_get(doc, '$.shipping.city', 'string') AS shipping_city,
           schema_of_variant(doc:shipping.city) AS shipping_city_type,
           try_variant_get(doc, '$.payment.method', 'string') AS payment_method,
           schema_of_variant(doc:payment.method) AS payment_method_type
    FROM nosql_orders_challenge WHERE order_id = 'O_NUEVO_001'
    """).first()
    new_lines = spark.sql("""
    SELECT try_variant_get(e.value, '$.product_id', 'string') AS product_id,
           try_variant_get(e.value, '$.category', 'string') AS category,
           try_variant_get(e.value, '$.quantity', 'bigint') AS quantity,
           try_variant_get(e.value, '$.unit_price', 'decimal(12,2)') AS price
    FROM nosql_orders_challenge AS o, LATERAL variant_explode(o.doc:items) AS e
    WHERE o.order_id = 'O_NUEVO_001'
    """).collect()
    validate_evolved_order(new, new_lines)
    print("Validación técnica OK. Las respuestas conceptuales se evalúan por separado.")


## Entrega
Exportá este notebook ejecutado con su control final aprobado. Completá el README usando
`PLANTILLA_ENTREGA.md`: identificación, escala y catálogo/esquema, resultados A/B,
evidencia C, respuestas D01–D05 y decisiones de diseño de 01.
Para los análisis, incluí consulta SQL/PySpark, resultado e interpretación.
Guardalos en `resolucion-nosql/` de tu repositorio personal, con capturas si faltan salidas.
Las soluciones docentes se consultan después de la entrega.
